# 15 — Aset gambar (CPU, plus satu unduhan citra)

Dua gambar yang belum bisa dibuat butuh masukan yang tidak dibawa laporan run.

| gambar | butuh | dari |
|---|---|---|
| metode, dengan citra asli | matriks bobot kepala, nama kelas, satu citra val per kelas | bagian 1 dan 2 di bawah |
| kurva coverage per kelas (apendiks) | coverage tiap kelas per lengan | `--dump-fit` yang sudah diperluas |

**Kelas yang ditampilkan ditetapkan aturan, bukan pilihan.** Aturannya: kelas held-out dengan
coverage **terendah** di bawah ambang marginal, yang menurut run primer adalah **kelas 754**.
Memilihnya setelah melihat tetangga mana yang paling terlihat mirip berarti memilih contoh
supaya cocok dengan argumen. Tetangganya diambil dari jarak cosine di ruang bobot kepala,
yaitu deskriptor yang dipakai metodenya.

Biaya: bagian 1 dan 3 hitungan detik. Bagian 2 mengunduh tar validasi ImageNet (6,3 GB) kalau
notebook 11 belum meninggalkannya di Drive, lalu **hanya mengekstrak citra yang dipakai** dan
membuang tarnya. Tidak ada GPU.

## 1. Config repo, unduhan, dan Drive

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 4. Dump, kepala, dan repo pesaing

In [ ]:
import glob

def _pair(scores):
    for suf in ('_softmax.npy', '_scores.npy', 'scores.npy'):
        if scores.endswith(suf):
            cand = scores[: -len(suf)] + suf.replace('softmax', 'labels').replace(
                'scores', 'labels')
            if os.path.exists(cand):
                return cand
    cand = os.path.join(os.path.dirname(scores), 'labels.npy')
    return cand if os.path.exists(cand) else None

DUMPS = {}

# CCC: materialisasi sel di atas menaruhnya di /content/ccc_npy/<ds>/
for p in sorted(glob.glob('/content/ccc_npy/*/scores.npy')):
    ds = os.path.basename(os.path.dirname(p))
    lab = _pair(p)
    if lab:
        DUMPS['ccc_' + ds] = {'scores': p, 'labels': lab, 'eval_scores': None,
                              'eval_labels': None, 'max_rows': MAX_ROWS}

# LTC: pasangkan cal (DESC+CAL) dengan test (EVAL penuh)
for ds in LTC_DATASETS:
    d = f'{DRIVE_ROOT}/released_scores/{ds}'
    cal = sorted(glob.glob(f'{d}/**/*cal_softmax.npy', recursive=True))
    tst = sorted(glob.glob(f'{d}/**/*test_softmax.npy', recursive=True))
    cal = [p for p in cal if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    tst = [p for p in tst if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    if cal and tst and _pair(cal[0]) and _pair(tst[0]):
        DUMPS['ltc_' + ds] = {'scores': cal[0], 'labels': _pair(cal[0]),
                              'eval_scores': tst[0], 'eval_labels': _pair(tst[0]),
                              'max_rows': None}

assert DUMPS, 'tidak ada dump siap pakai -- periksa sel penyiapan di atas'
for k, v in DUMPS.items():
    a = np.load(v['scores'], mmap_mode='r')
    line = '  ' + k.ljust(18) + ' cal ' + str(a.shape)
    if v['eval_scores']:
        e = np.load(v['eval_scores'], mmap_mode='r')
        line += '  eval ' + str(e.shape) + '  (dump terpisah)'
    else:
        line += '  eval = 30% dari dump yang sama'
    print(line)

In [ ]:
import numpy as np, os, glob, subprocess

HEAD_DIR = '/content/head'
os.makedirs(HEAD_DIR, exist_ok=True)
HEADS = {}          # K -> (path W, path b)

def _register(tag, W, b):
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    b_p = f'{HEAD_DIR}/{tag}_fc_bias.npy'
    np.save(w_p, W)
    np.save(b_p, np.zeros(len(W)) if b is None else b)
    HEADS[int(W.shape[0])] = (w_p, b_p)
    print('  kepala', tag, W.shape, '-> K =', W.shape[0])

# --- 1. torchvision ResNet-50 (ImageNet-1k). Model tersupervisi yang BERBEDA dari
# SimCLRv2+probe penghasil skor CCC: ketidakcocokan itu justru yang membuat phi
# eksogen sepenuhnya terhadap delta_y.
tv_w = f'{HEAD_DIR}/torchvision_imagenet_fc_weight.npy'
if os.path.exists(tv_w):
    W = np.load(tv_w)
    HEADS[int(W.shape[0])] = (tv_w, tv_w.replace('_weight', '_bias'))
    print('  kepala torchvision_imagenet sudah ada -> K =', W.shape[0])
else:
    from pcc.descriptors.head_weights import load_torchvision_resnet50_head
    W, b = load_torchvision_resnet50_head()
    _register('torchvision_imagenet', W, b)

# --- 2. Kepala LTC untuk Pl@ntNet dan iNat-2018. Run pertama Phase 2 gagal di kedua
# dataset itu, tetapi HANYA keluarga ruang-output yang pernah dijalankan di sana --
# dan keluarga itu juga gagal di ImageNet. Jadi itu kegagalan KELUARGA phi, bukan
# kegagalan dataset, dan checkpoint yang dirilis membuatnya bisa diuji tanpa GPU.
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip, dari notebook 00
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/ltc_models'
os.makedirs(CKPT_DIR, exist_ok=True)
if not glob.glob(f'{CKPT_DIR}/**/*model*.pth', recursive=True):
    print('mengunduh models.zip LTC (6 ResNet-50)...')
    subprocess.run(['gdown', GID_MODELS, '-O', f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip', '-o', f'{CKPT_DIR}/models.zip', '-d', CKPT_DIR],
                   check=True)

def _variant_ok(path):
    # PERANGKAP dari notebook 00: LTC mengirim ENAM model dengan NAMA BERKAS
    # IDENTIK dan menaruh varian focal di subdirektori 'focal_loss'. Glob rekursif
    # bisa mengambil mana saja, jadi checkpoint dan skor bisa diam-diam berasal dari
    # varian berbeda -- akurasi mirip, kalibrasi beda total.
    is_focal = 'focal' in path.replace(chr(92), '/').lower()
    return is_focal if LOSS_VARIANT == 'focal' else (not is_focal)

from pcc.data.ltc_datasets import NUM_CLASSES
for ds in LTC_DATASETS:
    tag = 'ltc_' + ds
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    if os.path.exists(w_p):
        W = np.load(w_p, mmap_mode='r')
        HEADS[int(W.shape[0])] = (w_p, w_p.replace('_weight', '_bias'))
        print('  kepala', tag, 'sudah ada -> K =', W.shape[0])
        continue
    cands = sorted(glob.glob(f'{CKPT_DIR}/**/best-{ds}-model.pth', recursive=True))
    keep = [p for p in cands if _variant_ok(p)]
    print('  checkpoint', ds, ':', len(cands), 'kandidat,', len(keep),
          'cocok varian', LOSS_VARIANT)
    for p in cands:
        print('     ' + ('* ' if _variant_ok(p) else '  ') + p)
    if not keep:
        print('     DILEWATI: tidak ada checkpoint varian', LOSS_VARIANT)
        continue
    try:
        from pcc.extract.backbones import load_ltc_resnet50
        m = load_ltc_resnet50(keep[0], NUM_CLASSES[ds], None)
        W = m.fc.weight.detach().cpu().numpy()
        b = m.fc.bias.detach().cpu().numpy() if m.fc.bias is not None else None
        del m
        assert W.shape[0] == NUM_CLASSES[ds], (W.shape, NUM_CLASSES[ds])
        _register(tag, W, b)
    except Exception as e:
        print('     GAGAL memuat:', type(e).__name__, str(e)[:160])

print()
print('kepala tersedia per jumlah kelas:', {k: os.path.basename(v[0])
                                            for k, v in sorted(HEADS.items())})

In [ ]:
# === bagian 1: jalankan ulang konfigurasi primer dengan --dump-fit yang diperluas ====
import subprocess, sys, os, numpy as np

PRIMARY = 'ccc_imagenet'
assert PRIMARY in DUMPS, 'dump primer tak ada: ' + repr(sorted(DUMPS))
PRIMARY_S, PRIMARY_Y = DUMPS[PRIMARY]['scores'], DUMPS[PRIMARY]['labels']
_K = int(np.load(PRIMARY_S, mmap_mode='r').shape[1])
assert _K in HEADS, 'tidak ada kepala untuk K=%d' % _K
HEAD_W, HEAD_B = HEADS[_K]
ASSETS = DRIVE_ROOT + '/runs/figassets'
os.makedirs(ASSETS, exist_ok=True)
FIT_NPZ = DRIVE_ROOT + '/runs/fit_primary.npz'
print('dump primer', PRIMARY, '| K =', _K, '| kepala', os.path.basename(HEAD_W))

cmd = [sys.executable, '-m', 'pcc.experiments.phase2_pcc',
       '--scores', PRIMARY_S, '--labels', PRIMARY_Y, '--dataset', PRIMARY,
       '--alpha', '0.10', '--n-cal', '25',
       '--heldout-frac', '0.30', '--frac-desc', '0.0', '--frac-cal', '0.70',
       '--max-rows', str(MAX_ROWS), '--phi', 'head', '--head-weights', HEAD_W,
       '--reports-dir', 'pcc/reports', '--seed', '0', '--dump-fit', FIT_NPZ]
if HEAD_B:
    cmd += ['--head-bias', HEAD_B]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode:
    print('GAGAL'); print(r.stderr[-2000:])
else:
    z = np.load(FIT_NPZ, allow_pickle=True)
    print('larik coverage:', sorted(k for k in z.files if k.startswith('cov_')))

In [ ]:
# === bagian 2: kelas mana, dan tetangganya di ruang bobot =============================
import numpy as np, json, os

W = np.load(HEAD_W).astype(np.float64)
Wn = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-12)
sim = Wn @ Wn.T
np.fill_diagonal(sim, -np.inf)

z = np.load(FIT_NPZ, allow_pickle=True)
held = np.asarray(z['heldout'], int)
cov_ho = np.asarray(z['cov_heldout_uncorrected'], float)   # sejajar dengan `held`
# ATURAN, ditetapkan sebelum melihat citra apa pun: kelas held-out dengan coverage terendah
target = int(held[int(np.nanargmin(cov_ho))])
nbrs = [int(i) for i in np.argsort(-sim[target])[:3]]
lam, q = float(z['lam']), float(z['q_global'])
dt = lam * float(z['delta_hat'][target])

print('kelas target (coverage held-out terendah):', target,
      '| coverage %.3f' % float(np.nanmin(cov_ho)))
print('tetangga terdekat di ruang bobot         :', nbrs)
print('  jarak cosine:', [round(1 - float(sim[target, n]), 4) for n in nbrs])
print('ambang: q_hat %.4f -> q_hat + delta_tilde %.4f  (delta_tilde %+.4f)' % (q, q + dt, dt))

# nama kelas ImageNet, dari torchvision kalau ada, kalau tidak dari indeks wnid
try:
    from torchvision.models import ResNet50_Weights
    NAMES = list(ResNet50_Weights.IMAGENET1K_V1.meta['categories'])
except Exception as e:
    print('nama kelas torchvision tidak tersedia:', e)
    NAMES = ['class %d' % i for i in range(_K)]
meta = {'target': target, 'neighbours': nbrs,
        'names': {str(c): NAMES[c] for c in [target] + nbrs},
        'cosine_distance': {str(n): 1 - float(sim[target, n]) for n in nbrs},
        'coverage_uncorrected': float(np.nanmin(cov_ho)),
        'q_global': q, 'lam': lam, 'delta_tilde': dt,
        'rule': 'held-out class with the lowest coverage under the marginal threshold'}
json.dump(meta, open(ASSETS + '/meta.json', 'w'), indent=1)
print()
for c in [target] + nbrs:
    print('  %4d  %s' % (c, NAMES[c]))
print('ditulis:', ASSETS + '/meta.json')

In [ ]:
# === bagian 3: satu citra validasi per kelas ==========================================
# Hanya empat citra yang dipakai. Tar validasi diunduh HANYA kalau notebook 11 belum
# meninggalkan foldernya, dan yang diekstrak cuma anggota yang dibutuhkan.
import os, json, subprocess, tarfile, urllib.request, glob, shutil

meta = json.load(open(ASSETS + '/meta.json'))
WANT = [meta['target']] + meta['neighbours']

VAL_DIR = '/content/imagenet_val'
# satu wnid per citra validasi, berurutan menurut indeks berkas; sumber yang sama
# dipakai notebook 11
LBL_URL = ('https://raw.githubusercontent.com/tensorflow/models/master/research/'
           'slim/datasets/imagenet_2012_validation_synset_labels.txt')

def val_wnids():
    return urllib.request.urlopen(LBL_URL).read().decode().split()

got = {}
if os.path.isdir(VAL_DIR) and glob.glob(VAL_DIR + '/*/*.JPEG'):
    print('memakai folder val yang sudah ada dari notebook 11')
    classes = sorted(os.listdir(VAL_DIR))
    for c in WANT:
        files = sorted(glob.glob(os.path.join(VAL_DIR, classes[c], '*.JPEG')))
        if files:
            got[c] = files[0]
else:
    print('folder val tidak ada; mengunduh tar dan mengekstrak hanya yang dipakai')
    val_labels = val_wnids()
    classes = sorted(set(val_labels))
    want_wnid = {classes[c]: c for c in WANT}
    # indeks citra val (1-based) untuk tiap kelas yang dipakai, ambil yang pertama
    first = {}
    for i, w in enumerate(val_labels, start=1):
        if w in want_wnid and w not in first:
            first[w] = 'ILSVRC2012_val_%08d.JPEG' % i
    print('  berkas yang dicari:', sorted(first.values()))
    TAR = '/content/val.tar'
    if not os.path.exists(TAR):
        subprocess.run(['wget', '-q', '-O', TAR,
                        'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar'],
                       check=True)
    need = set(first.values())
    with tarfile.open(TAR) as tf:
        for m in tf:
            if m.name in need:
                w = [k for k, v in first.items() if v == m.name][0]
                c = want_wnid[w]
                with open(ASSETS + '/cls%d.JPEG' % c, 'wb') as fh:
                    shutil.copyfileobj(tf.extractfile(m), fh)
                got[c] = ASSETS + '/cls%d.JPEG' % c
                need.discard(m.name)
                if not need:
                    break
    os.remove(TAR)

# kecilkan ke 224x224 supaya asetnya ringan
from PIL import Image
for c, src in sorted(got.items()):
    dst = ASSETS + '/cls%d.jpg' % c
    im = Image.open(src).convert('RGB')
    s = min(im.size)
    im = im.crop(((im.width - s) // 2, (im.height - s) // 2,
                  (im.width + s) // 2, (im.height + s) // 2)).resize((224, 224))
    im.save(dst, quality=88)
    print('  %-24s %s' % (os.path.basename(dst), meta['names'][str(c)]))
for f in glob.glob(ASSETS + '/*.JPEG'):
    os.remove(f)
print()
print('aset di', ASSETS, ':', sorted(os.listdir(ASSETS)))

## Yang kutunggu

Salin **seluruh folder** `DRIVE_ROOT/runs/figassets/` ke akar repo sebagai `figassets/`, dan
`fit_primary.npz` yang baru (sekarang membawa larik coverage per kelas). Lalu bilang padaku.

Kalau bagian 3 gagal karena tar validasi tidak bisa diunduh, bilang saja — gambar metodenya
tetap bisa dibuat tanpa citra, cuma kehilangan bagian yang kau minta.